# C3 Volatility-Scaled SL/TP — Phase 1
Frozen global ATR20 primary and RV20 robustness rules. See `docs/91_c3_volatility_scaled_sltp_phase1_plan.md`.

## 1. Source paths
Provide the frozen baseline and audited 56-file M1 root. Output goes to `/content`; Drive saving is OFF by default.

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess, sys
BASELINE = Path('/content/daily_stop_baseline_trades.csv')
M1_ROOT = Path('/content/m1')
OUT = Path('/content')
DRIVE_SAVE = False
assert BASELINE.exists() and M1_ROOT.exists(), 'Provide the audited inputs before running.'

## 2. R0 hard gate
The command stops before C3 outcomes if any baseline or M1 audit check fails.

In [ ]:
runner = Path('src/research/c3_vol_scaled_sltp_phase1.py')
manifest = Path('results/volatility_phase1/volatility_phase1_input_manifest.csv')
subprocess.run([sys.executable, str(runner), '--baseline', str(BASELINE), '--manifest', str(manifest), '--m1-root', str(M1_ROOT), '--out', str(OUT), '--stage', 'reconcile'], check=True)
display(pd.read_csv(OUT/'c3_vol_scaled_sltp_phase1_r0_reconciliation.csv'))
display(pd.read_csv(OUT/'c3_vol_scaled_sltp_phase1_m1_audit.csv').head())

## 3–10. Primary, robustness, periods, scale, diagnostics and gates
Run only after both R0 scopes show zero mismatches.

In [ ]:
subprocess.run([sys.executable, str(runner), '--baseline', str(BASELINE), '--manifest', str(manifest), '--m1-root', str(M1_ROOT), '--out', str(OUT), '--stage', 'full'], check=True)
for title, suffix in [('Portfolio','portfolio_summary'), ('Periods and CI','period_summary'), ('Feature coverage','coverage'), ('Scale distribution','scale_distribution'), ('Trade deltas','trade_delta_summary'), ('Strategy diagnostics','strategy_summary'), ('Formal gates','formal_gates')]:
    print('\n' + title)
    frame = pd.read_csv(OUT/f'c3_vol_scaled_sltp_phase1_{suffix}.csv')
    display(frame.head(40) if suffix in ('scale_distribution','strategy_summary') else frame)

## 11. Verdict and Phase 2 eligibility
Only `VOLATILITY_SCALED_SLTP_CANDIDATE` may enter a separately preregistered Phase 2.

In [ ]:
formal = pd.read_csv(OUT/'c3_vol_scaled_sltp_phase1_formal_gates.csv')
display(formal)
print('Phase 2:', 'ELIGIBLE FOR SEPARATE PLAN' if formal.Label.eq('VOLATILITY_SCALED_SLTP_CANDIDATE').any() else 'DO NOT PROCEED')
if DRIVE_SAVE:
    raise NotImplementedError('Drive save is deliberately OFF; review destination and artifacts before enabling.')